# 🚀 This project demonstrates data enrichment using web scraping and prepares a dataset for geospatial visualization in Power BI.

# 📊 MP Profile Image Scraper (Data Augmentation Pipeline)

## 🎯 Objective
This notebook extracts profile image URLs of Members of Parliament (MPs) from the PRS India website and enriches the dataset for further analysis and visualization.

---

## 📂 Dataset
- Source: PRS India MP Track dataset
- File: `18 LS MP Track.csv`
- Contains MP details such as name, party, state, etc.

---

## ⚙️ Approach

This project follows a data enrichment pipeline:

1. **Data Cleaning**
   - Convert MP names into URL-compatible format
   - Handle inconsistencies in naming (spaces, punctuation, casing)

2. **Handling Real-World Data Issues**
   - PRS URLs are not standardized
   - Manual mapping is added for mismatched names
   - Ensures higher accuracy (~100%)

3. **Web Scraping (HTML Parsing)**
   - Fetch MP profile pages using `requests`
   - Parse HTML using `BeautifulSoup`

4. **DOM Inspection & CSS Selectors**
   - Identify image location using browser DevTools
   - Use selector:
     ```
     div.field-item img.img-responsive
     ```

5. **Data Extraction**
   - Extract image `src` attribute
   - Convert relative path → full URL

6. **Output Generation**
   - Store image URLs in `imageLink` column
   - Export enriched dataset for Power BI dashboard

---

## 🧠 Key Learnings

- Real-world data is often inconsistent
- Web scraping requires DOM understanding
- CSS selectors must match actual webpage structure
- Hybrid approach (automation + manual fixes) is often necessary

---

## 🚀 Output

- Enriched dataset: `MPwithImageLink.csv`
- Ready to be used in Power BI for dashboard creation

In [18]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

In [19]:
# ✅ Load dataset
dataset = pd.read_csv("18 LS MP Track.csv")

# ✅ Create column for image links
dataset["imageLink"] = None

In [20]:
# ✅ Manual corrections for PRS URL mismatches (https://prsindia.org/mptrack/18th-lok-sabha)
# ------------------------------------------------------------------------------------------------------------------------------------------
# This step ensures nearly 100% accuracy in mapping dataset names to PRS profile URLs.

# Why this step is required:
#
# The PRS India website does not follow a consistent naming convention
# for MP profile URLs. While most URLs can be generated by cleaning
# 'mp_name' and replacing spaces with hyphens, several edge cases exist:
#
# 1. Middle names are removed
#    Example:
#    "Shrirang Appa Chandu Barne" → "shrirang-appa-barne"
#
# 2. Words are split differently
#    Example:
#    "Bhupathiraju" → "bhupathi-raju"
#
# 3. Prefixes are added (e.g., "adv-", "dr-", "captain-")
#    Example:
#    "Gowaal Kagada Padavi" → "adv-gowaal-kagada-padavi"
#
# 4. Spelling variations occur
#    Example:
#    "Chandolia" → "Chandoliya"
#
# Because of these inconsistencies, automatic URL generation does not achieve 100% accuracy. Therefore, a manual mapping dictionary is used
# to correct known mismatches and ensure complete data coverage.
#
# This hybrid approach (automation + manual correction) reflects real-world data engineering practices.
# ------------------------------------------------------------------------------------------------------------------------------------------

name_fix = {
    "yogender chandolia": "yogender-chandoliya",
    "gowaal kagada padavi": "adv-gowaal-kagada-padavi",
    "shrirang appa chandu barne": "shrirang-appa-barne",
    "bhupathiraju srinivasa varma": "bhupathi-raju-srinivasa-varma",
    "g m harish balayogi": "ganti-harish-madhur",
    "kinjarapu rammohan naidu": "ram-mohan-naidu-kinjarapu",
    "rajiv ranjan singh": "rajiv-ranjan-singh-alias-lalan-singh",
    "viriato fernandes": "captain-viriato-fernandes",
    "amit shah": "amit-anil-chandra-shah",
    "chandrakant raghunath patil": "c-r-patil",
    "nimuben jayantibhai bambhaniya": "nimuben-jayantibhai-bambhaniya-nimuben-bambhaniya",
    "rao inderjit singh": "inderjit-singh-rao",
    "krishan pal gurjar": "krishan-pal",
    "manohar lal": "manohar-lal-khattar",
    "pralhad joshi": "pralhad-venkatesh-joshi",
    "durgadas d d uikey": "durga-das-d-d-uikey",
    "jyotiraditya m scindia": "jyotiraditya-madhavrao-scindia",
    "arvind ganpat sawant": "arvind-sawant",
    "prashant yadaorao padole": "dr-prashant-yadaorao-padole",
    "prataprao ganpatrao jadhav": "jadhav-prataprao-ganpatrao",
    "omprakash bhupalsinh alias pavan rajenimbalkar": "omprakash",
    "piyush vedprakash goyal": "piyush-goyal",
    "k gopinath": "gopinath-k",
    "raksha nikhil khadse": "raksha-nikhil-khadase",
    "vishaldada prakashbapu patil": "vishal-dada-prakashbapu-patil",
    "damodar agrawal": "damodar-agarwal",
    "lumba ram": "lumbaram",
    "rajkumar roat": "raj-kumar-roat",
    "t sumathy alias thamizhachi thangapandian": "t-sumathy",
    "kalanidhi veeraswamy": "veeraswamy-kalanidhi",
    "bandi sanjay kumar": "sanjay-kumar-bandi",
    "anand bhadauria": "anand-bhadauriya",
    "chandra shekhar": "chandrashekhar",
    "pankaj choudhary": "pankaj-chaudhary",
    "muhammed hamdullah sayeed": "hamdullah-sayeed",
    "rajesh ranjan": "rajesh-ranjan-alias-pappu-yadav",
    "ricky andrew j syngkon": "dr-ricky-andrew-j-syngkon",
    "delkar kalaben mohanbhai": "kalaben-mohanbhai-delkar",
    "g lakshminarayana": "ambica-g-lakshminarayana-valmiki",
    "daggubati purandeswari": "daggubati-purandheshwari",
    "kesineni sivanath": "kesineni-sivanath-chinni",
    "magunta sreenivasulu reddy": "magunta-srinivasulu-reddy",
    "bastipati nagaraju": "nagaraju",
    "sribharat mathukumilli": "sribharat-mathukumili",
    "md. rakibul hussain": "rakibul-hussain",
    "roopkumari choudhary": "roop-kumari-choudhary",
    "chavda vinod lakhamshi": "chavda-vinod-lakhamashi",
    "bharatsinhji shankarji dabhi": "dabhi-bharatsinhji-shankarji",
    "hasmukhbhai somabhai patel": "hasmukhbhai-patel",
    "jaswantsinh sumanbhai bhabhor": "jasvantsinh-sumanbhai-bhabhor",
    "mansukhbhai dhanjibhai vasava": "mansukhbhai-d-vasava",
    "mitesh patel bakabhai": "mitesh-rameshbhai-patel",
    "kumari selja": "selja",
    "varun chaudhry": "varun-chaudhury",
    "brijesh chowta": "captain-brijesh-chowta",
    "c n manjunath": "dr-c-n-manjunath",
    "kota srinivasa poojary": "kota-srinivas-poojary",
    "gaddigoudar parvatagouda chandanagouda": "p-c-gaddigoudar",
    "vishweshwar hegde kageri": "vishweshwar-hegde",
    "yaduveer wadiyar": "yaduveer-krishnadatta-chamaraja-wadiyar",
    "k sudhakaran": "kumbakudi-sudhakaran",
    "kodikunnil suresh": "suresh-kodikunnil",
    "rajesh mishra": "dr-rajesh-mishra",
    "gajendra singh patel": "gajendra-umrao-singh-patel",
    "sudheer gupta": "sudhir-gupta",
    "dhanorkar pratibha suresh": "dhanorkar-pratibha-suresh-alias-balubhau",
    "shivaji bandappa kalge": "dr-kalge-shivaji-bandappa",
    "rajabhau parag prakash waje": "parag-prakash-waje",
    "shyamkumar daulat barve": "shyamkumar-babalu-daulat-barve",
    "balya mama suresh gopinath mhatre": "suresh-gopinath-mhatre",
    "tatkare sunil dattatrey": "tatkare-sunil-dattatray",
    "pradeep kumar panigrahy": "dr-pradeep-kumar-panigrahy",
    "amra ram": "amraram",
    "raja a": "a-raja",
    "d ravi kumar": "d-ravikumar",
    "rani srikumar": "dr-rani-sri-kumar",
    "selvam g": "g-selvam",
    "subbarayan k": "k-subbarayan",
    "s jagathratchakan": "s-jagathrakshakan",
    "selvaganapathi t.m.": "t-m-selvaganapathi",
    "t r baalu": "thalikkottai-rajuthevar-baalu",
    "tamilselvan thanga": "thanga-tamil-selvan",
    "thirumaavalavan tholkappiyan": "thirumaavalavan-thol",
    "d k aruna": "aruna-d-k",
    "rajkumar sangwan": "dr-rajkumar-sangwan",
    "anoop pradhan valmiki": "anoop-pradhan-balmiki",
    "virendra singh": "birendra-singh",
    "chhatrapal singh gangwar": "chhatra-pal-singh-gangwar",
    "vinod kumar bind": "dr-vinod-kumar-bind",
    "hemamalini dharmendra deol": "hema-malini",
    "krishna devi shivshankar patel": "krishna-devi-shivshanker-patel",
    "narayandas ahirwar": "narayan-das-ahirwar",
    "satish kumar gautam": "satish-kumar",
    "shiv pal singh patel": "shiv-pal-singh-patel-dr-s-p-singh",
    "vijay kumar dubey": "vijay-kumar-dubay",
    "adhikari deepak dev": "deepak-dev-adhikari",
    "shatrughan prasad sinha": "shatrughan-sinha",
    "chh. udayanraje pratapsinha maharaj bhonsle": "shrimant-chh-udyanraje-pratapsinhmaharaj-bhonsle",
    "swami sachidanand hari sakshi": "swami-sachchidanand-hari-sakshi"
}

In [21]:
# ✅ -------------------------------------------------------------
# Main Data Enrichment Loop: Extract MP profile images from PRS
# -------------------------------------------------------------
#
# 📊 Objective:
# Enrich the existing MP dataset by adding profile image URLs
# from the PRS India website.
#
# 🧠 Approach:
# This process combines:
# - Data cleaning (to generate URL slugs)
# - HTML parsing (to extract image elements)
# - CSS selector targeting (to locate correct image node)
#
# -------------------------------------------------------------
# ✅ Step 1: Name Cleaning (URL Slug Generation)
#
# PRS profile pages use URL slugs based on MP names.
# Example:
#   "Manoj Tiwari" → "manoj-tiwari"
#
# However, inconsistencies exist:
# - Middle names may be removed
# - Prefixes may be added (dr-, adv-, captain-)
# - Spellings may differ slightly
#
# Therefore:
# - We first normalize names (lowercase, remove special characters)
# - Then convert spaces → hyphens
# - Then apply manual fixes for known mismatches
#
# -------------------------------------------------------------
# ✅ Step 2: URL Construction
#
# Each MP profile follows:
# https://prsindia.org/mptrack/18th-lok-sabha/{slug}
#
# Example:
# https://prsindia.org/mptrack/18th-lok-sabha/manoj-tiwari
#
# -------------------------------------------------------------
# ✅ Step 3: HTML Parsing
#
# The page is fetched using requests and parsed using BeautifulSoup.
# We inspect the DOM to identify where the profile image is located.
#
# -------------------------------------------------------------
# ✅ Step 4: CSS Selector (CRITICAL)
#
# From DevTools inspection, the MP image is located at:
#
# div.field-item → img.img-responsive
#
# So we use:
# soup.select("div.field-item img.img-responsive")
#
# This ensures we capture the actual profile image instead of
# logos or unrelated images.
#
# -------------------------------------------------------------
# ✅ Step 5: Extract Image URL
#
# Extract 'src' attribute from <img> tag:
# Example:
# /files/mptrack/18-lok-sabha/profile_image/180098.jpg
#
# Convert to full URL:
# https://prsindia.org/files/mptrack/18-lok-sabha/profile_image/180098.jpg
#
# -------------------------------------------------------------
# ✅ Step 6: Save Results
#
# Store extracted imageLink back into dataset.
#
# -------------------------------------------------------------
# ✅ Real-World Learning:
#
# - Web data is rarely standardized
# - Hybrid approach (automation + manual fixes) is often required
# - DOM inspection is essential for correct scraping
#
# -------------------------------------------------------------

In [22]:
# ✅ LOOP through each MP
for i in range(len(dataset)):

    try:
        # ✅ Extract original MP name
        original_name = dataset.loc[i, "mp_name"]

        # ✅ Normalize name (lowercase + remove extra spaces)
        name_clean = str(original_name).lower().strip()

        # ✅ Apply manual correction if exists (handles edge cases)
        if name_clean in name_fix:
            name = name_fix[name_clean]
        else:
            # ✅ Default cleaning → convert to URL slug
            name = re.sub(" ", "-", name_clean)
            name = re.sub(r"\.", "", name)
            name = re.sub(r"\(", "", name)
            name = re.sub(r"\)", "", name)
            name = re.sub(r"\@", "", name)

        # ✅ Construct profile URL
        link = f"https://prsindia.org/mptrack/18th-lok-sabha/{name}"

        print(f"🔗 {link}")  # debugging

        # ✅ Fetch page HTML
        response = requests.get(
            link,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=5
        )

        # ✅ Skip invalid URLs
        if response.status_code != 200:
            print(f"❌ Not found: {original_name}")
            continue

        # ✅ Parse HTML
        soup = BeautifulSoup(response.content, "html.parser")

        # ✅ Extract image using CSS selector
        img_tags = soup.select("div.field-item img.img-responsive")

        imageLink = None

        if len(img_tags) > 0:

            # ✅ Get image source
            src = img_tags[0].get("src")

            # ✅ Convert relative path → full URL
            if src.startswith("/"):
                imageLink = "https://prsindia.org" + src
            else:
                imageLink = src

            imageLink = imageLink.replace(" ", "")

            print(f"✅ {i+1}/{len(dataset)} → {original_name}")

        else:
            print(f"❌ No image: {original_name}")

        # ✅ Save imageLink in dataset
        dataset.loc[i, "imageLink"] = imageLink

        # ✅ Delay to avoid blocking
        time.sleep(0.5)

    except Exception as e:
        print(f"⚠ Error at: {original_name}")

🔗 https://prsindia.org/mptrack/18th-lok-sabha/harsh-malhotra
✅ 1/544 → Harsh Malhotra
🔗 https://prsindia.org/mptrack/18th-lok-sabha/manoj-tiwari
✅ 2/544 → Manoj Tiwari
🔗 https://prsindia.org/mptrack/18th-lok-sabha/kamaljeet-sehrawat
✅ 3/544 → Kamaljeet Sehrawat
🔗 https://prsindia.org/mptrack/18th-lok-sabha/praveen-khandelwal
✅ 4/544 → Praveen Khandelwal
🔗 https://prsindia.org/mptrack/18th-lok-sabha/ramvir-singh-bidhuri
✅ 5/544 → Ramvir Singh Bidhuri
🔗 https://prsindia.org/mptrack/18th-lok-sabha/yogender-chandoliya
✅ 6/544 → Yogender Chandolia
🔗 https://prsindia.org/mptrack/18th-lok-sabha/adv-gowaal-kagada-padavi
✅ 7/544 → Gowaal Kagada Padavi
🔗 https://prsindia.org/mptrack/18th-lok-sabha/shrirang-appa-barne
✅ 8/544 → Shrirang Appa Chandu Barne
🔗 https://prsindia.org/mptrack/18th-lok-sabha/sukanta-majumdar
✅ 9/544 → Sukanta Majumdar
🔗 https://prsindia.org/mptrack/18th-lok-sabha/bhupathi-raju-srinivasa-varma
✅ 10/544 → Bhupathiraju Srinivasa Varma
🔗 https://prsindia.org/mptrack/18th-lok-

In [23]:
# ✅ Preview final dataset
# This step is used to verify that the imageLink column has been correctly populated
# and to quickly inspect the first few records for validation.

dataset[["mp_name", "imageLink"]].head(10)

,mp_name,imageLink
0,Harsh Malhotra,https://prsindia.org/files/mptrack/18-lok-sabh...
1,Manoj Tiwari,https://prsindia.org/files/mptrack/18-lok-sabh...
2,Kamaljeet Sehrawat,https://prsindia.org/files/mptrack/18-lok-sabh...
3,Praveen Khandelwal,https://prsindia.org/files/mptrack/18-lok-sabh...
4,Ramvir Singh Bidhuri,https://prsindia.org/files/mptrack/18-lok-sabh...
5,Yogender Chandolia,https://prsindia.org/files/mptrack/18-lok-sabh...
6,Gowaal Kagada Padavi,https://prsindia.org/files/mptrack/18-lok-sabh...
7,Shrirang Appa Chandu Barne,https://prsindia.org/files/mptrack/18-lok-sabh...
8,Sukanta Majumdar,https://prsindia.org/files/mptrack/18-lok-sabh...
9,Bhupathiraju Srinivasa Varma,https://prsindia.org/files/mptrack/18-lok-sabh...


In [24]:
# ✅ Save output
# Export the enriched dataset (with image URLs) to a CSV file.
# This file will be used later in Power BI for visualization and dashboard creation.
dataset.to_csv("MPwithImageLink.csv", index=False)

print("✅ DONE → File saved as MPwithImageLink.csv")

✅ DONE → File saved as MPwithImageLink.csv
